In [1]:
import os

NEWS_API_KEY = "b3c66662c4544ed7beea105061d8e3b4"



if NEWS_API_KEY and NEWS_API_KEY != "b3c66662c4544ed7beea105061d8e3b4":
    print("API key loaded successfully.")
else:
    print("WARNING: Replace 'your_actual_newsapi_key_here' with your real key.")

In [2]:
print(repr(NEWS_API_KEY))

'b3c66662c4544ed7beea105061d8e3b4'


## Step 3 — NewsAPI Fetcher
### `ingestion/newsapi_fetcher.py`

In [3]:
import requests

def fetch_newsapi():
    """
    Fetch top headlines from NewsAPI.
    Uses 'sources=bbc-news' for free tier compatibility.
    Returns a list of article dicts.
    """
    if not NEWS_API_KEY:
        print("NewsAPI Error: API key not found in .env")
        return []

    url = (
        f"https://newsapi.org/v2/top-headlines?"
        f"sources=bbc-news&apiKey={NEWS_API_KEY}"
    )

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            articles = response.json().get("articles", [])
            print(f"NewsAPI: fetched {len(articles)} articles.")
            return articles
        else:
            msg = response.json().get("message", "Unknown error")
            print(f"NewsAPI Error {response.status_code}: {msg}")
            return []
    except requests.exceptions.RequestException as e:
        print(f"NewsAPI request failed: {e}")
        return []

## Step 4 — RSS Feed Fetcher
### `ingestion/rss_fetcher.py`

In [4]:
!pip install feedparser

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import feedparser

def fetch_rss(rss_url="http://feeds.bbci.co.uk/news/rss.xml"):
    """
    Fetch articles from an RSS feed.
    Defaults to BBC News. Pass any valid RSS URL to override.
    Returns a list of article dicts.
    """
    try:
        feed = feedparser.parse(rss_url)

        if feed.bozo and not feed.entries:
            print(f"RSS Error: Could not parse feed at {rss_url}")
            return []

        articles = []
        for entry in feed.entries:
            articles.append({
                "title":     entry.get("title", ""),
                "link":      entry.get("link", ""),
                "published": entry.get("published", "")
            })

        print(f"RSS: fetched {len(articles)} articles.")
        return articles

    except Exception as e:
        print(f"RSS request failed: {e}")
        return []

## Step 5 — GDELT Fetcher
### `ingestion/gdelt_fetcher.py`

In [6]:
import requests

def fetch_gdelt(query="technology", maxrecords=10):
    """
    Fetch articles from the GDELT Project API v2.
    No API key required.
    Returns a list of article dicts.
    """
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query":      query,
        "mode":       "ArtList",
        "maxrecords": maxrecords,
        "format":     "json"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code == 200:
            articles = response.json().get("articles", [])
            print(f"GDELT: fetched {len(articles)} articles.")
            return articles
        else:
            print(f"GDELT Error: {response.status_code}")
            return []
    except requests.exceptions.RequestException as e:
        print(f"GDELT request failed: {e}")
        return []

## Step 6 — Data Cleaning Module
### `processing/cleaner.py`

> **Bug fixed:** Original used `\\S` and `\\s` (literal backslashes). Correct regex requires raw strings `r"..."` so `\S` and `\s` work as metacharacters.

In [7]:
import re

def clean_text(text):
    """
    Clean article text:
    - Returns empty string for None/empty input
    - Removes URLs
    - Removes special characters (keeps letters, digits, spaces)
    - Strips leading/trailing whitespace
    """
    if not text:
        return ""

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def deduplicate(articles):
    """
    Remove duplicate articles based on title.
    Keeps first occurrence.
    """
    seen = set()
    unique = []
    for article in articles:
        title = article.get("title", "").lower().strip()
        if title and title not in seen:
            seen.add(title)
            unique.append(article)
    return unique

## Step 7 — Main Pipeline Script
### `main.py`

In [8]:
import json
import os

def save_articles(data, path="data/raw_articles.json"):
    """Save articles list to a JSON file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"Saved {len(data)} articles to {path}")


def main():
    global all_articles
    print("=" * 50)
    print(" Real-Time News Intelligence Agent — Week 1")
    print("=" * 50)

    print("\n[1/3] Fetching from NewsAPI...")
    newsapi_articles = fetch_newsapi()

    print("\n[2/3] Fetching from RSS (BBC News)...")
    rss_articles = fetch_rss()

    print("\n[3/3] Fetching from GDELT...")
    gdelt_articles = fetch_gdelt()

    all_articles = []

    for article in newsapi_articles:
        title = clean_text(article.get("title", ""))
        if title:
            all_articles.append({
                "title":        title,
                "source":       "NewsAPI",
                "description":  clean_text(article.get("description", "") or ""),
                "content":      clean_text(article.get("content", "") or ""),
                "url":          article.get("url", "") or "",
                "published_at": article.get("publishedAt", "") or "",
            })

    for article in rss_articles:
        title = clean_text(article.get("title", ""))
        if title:
            all_articles.append({
                "title":        title,
                "source":       "RSS",
                "description":  "",
                "content":      "",
                "url":          article.get("link", "") or "",
                "published_at": article.get("published", "") or "",
            })

    for article in gdelt_articles:
        title = clean_text(article.get("title", ""))
        if title:
            all_articles.append({
                "title":        title,
                "source":       "GDELT",
                "description":  "",
                "content":      "",
                "url":          article.get("url", "") or "",
                "published_at": article.get("seendate", "") or "",
            })

    all_articles = deduplicate(all_articles)

    save_articles(all_articles)

    print("\n" + "=" * 50)
    print(f" Total articles (after dedup): {len(all_articles)}")
    print(f"   NewsAPI : {sum(1 for a in all_articles if a['source'] == 'NewsAPI')}")
    print(f"   RSS     : {sum(1 for a in all_articles if a['source'] == 'RSS')}")
    print(f"   GDELT   : {sum(1 for a in all_articles if a['source'] == 'GDELT')}")
    print("=" * 50)

    print("\nSample output (first 5 articles):")
    for a in all_articles[:5]:
        print(f"  [{a['source']}] {a['title']}")

    return all_articles


all_articles = main()

 Real-Time News Intelligence Agent — Week 1

[1/3] Fetching from NewsAPI...
NewsAPI: fetched 10 articles.

[2/3] Fetching from RSS (BBC News)...
RSS: fetched 35 articles.

[3/3] Fetching from GDELT...
GDELT: fetched 10 articles.
Saved 51 articles to data/raw_articles.json

 Total articles (after dedup): 51
   NewsAPI : 10
   RSS     : 35
   GDELT   : 6

Sample output (first 5 articles):
  [NewsAPI] German broadcaster removes TV intro after Elon Musk takes legal action
  [NewsAPI] Shaftesbury Theatre in West End to be named after Dame Judi Dench
  [NewsAPI] MoD investigating reports Russian warship fired warning shots near yacht in Channel
  [NewsAPI] Three reasons ships are not sailing through the Strait of Hormuz yet
  [NewsAPI] Armed forces face cuts without more funding warns defence chief


## Step 8 — Unit Tests
### `tests/test_ingestion.py`

In [9]:

def test_clean_text():
    assert clean_text("") == ""
    assert clean_text(None) == ""
    assert clean_text("Hello, World!") == "Hello World"
    assert clean_text("Check https://example.com now!") == "Check now"
    assert clean_text("  spaces  ") == "spaces"
    print("clean_text: all tests passed.")


def test_deduplicate():
    articles = [
        {"title": "AI news", "source": "RSS"},
        {"title": "AI news", "source": "GDELT"},   # duplicate
        {"title": "New policy", "source": "NewsAPI"},
    ]
    result = deduplicate(articles)
    assert len(result) == 2, f"Expected 2, got {len(result)}"
    assert result[0]["title"] == "AI news"
    assert result[1]["title"] == "New policy"
    print("deduplicate: all tests passed.")


def test_fetch_rss_returns_list():
    result = fetch_rss()
    assert isinstance(result, list), "fetch_rss should return a list"
    if result:
        assert "title" in result[0], "Each article should have a 'title' key"
        assert "link" in result[0], "Each article should have a 'link' key"
    print(f"fetch_rss: returned {len(result)} articles — test passed.")


def test_fetch_gdelt_returns_list():
    result = fetch_gdelt()
    assert isinstance(result, list), "fetch_gdelt should return a list"
    print(f"fetch_gdelt: returned {len(result)} articles — test passed.")


print("Running Week 1 tests...\n")
test_clean_text()
test_deduplicate()
test_fetch_rss_returns_list()
test_fetch_gdelt_returns_list()
print("\nAll tests passed!")

Running Week 1 tests...

clean_text: all tests passed.
deduplicate: all tests passed.
RSS: fetched 35 articles.
fetch_rss: returned 35 articles — test passed.
GDELT: fetched 10 articles.
fetch_gdelt: returned 10 articles — test passed.

All tests passed!


## Step 9 — Generate `requirements.txt`

In [10]:
requirements = """requests==2.31.0
feedparser==6.0.12
python-dotenv==1.0.0
pandas==2.0.3
numpy==1.24.3
kafka-python==2.0.2
pyspark==3.4.1
fastapi==0.103.1
uvicorn==0.23.2
langchain==0.0.300
faiss-cpu==1.7.4
sentence-transformers==2.2.2
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created:")
print(requirements)

requirements.txt created:
requests==2.31.0
feedparser==6.0.12
python-dotenv==1.0.0
pandas==2.0.3
numpy==1.24.3
kafka-python==2.0.2
pyspark==3.4.1
fastapi==0.103.1
uvicorn==0.23.2
langchain==0.0.300
faiss-cpu==1.7.4
sentence-transformers==2.2.2



## Step 10 — Week 1 Deliverables Checklist

In [11]:
import os

checks = {
    "data/raw_articles.json exists":  os.path.exists("data/raw_articles.json"),
    "requirements.txt exists":        os.path.exists("requirements.txt"),
    "NEWS_API_KEY loaded":            bool(NEWS_API_KEY),
    "feedparser importable":          True,
    "requests importable":            True,
}

print("Week 1 Deliverables Checklist")
print("=" * 40)
all_good = True
for item, status in checks.items():
    icon = "✔" if status else "✘"
    print(f"  {icon}  {item}")
    if not status:
        all_good = False

print("=" * 40)
if all_good:
    print("All checks passed. Ready for Week 2!")
else:
    print("Some checks failed. See above.")

Week 1 Deliverables Checklist
  ✔  data/raw_articles.json exists
  ✔  requirements.txt exists
  ✔  NEWS_API_KEY loaded
  ✔  feedparser importable
  ✔  requests importable
All checks passed. Ready for Week 2!


# Week 2 — Data Ingestion & Processing with Kafka and PySpark

## Step 11 — Install Week 2 Dependencies

In [12]:
!pip install kafka-python pyspark findspark

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 12 — Initialize Spark

In [13]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("RealTimeNewsIntelligence")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2"
    )
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionString())
print("✔ Ready!")

Spark Version: 4.1.2
Scala: version 2.13.17
✔ Ready!


## Step 13 — Start Kafka in Colab
> **Important:** Kafka must be running before the producer can connect.
> This cell downloads, installs, and starts Kafka locally inside Colab.
> Run this cell once and wait for "Kafka is ready!" before proceeding.

In [15]:
import subprocess, time, os, requests, tarfile, tempfile, glob

KAFKA_VERSION = "3.6.1"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_TGZ     = f"{KAFKA_DIR}.tgz"

MIRRORS = [
    f"https://downloads.apache.org/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}",
    f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}",
]

# ---------- Download + extract ----------
if not os.path.isdir(KAFKA_DIR):
    downloaded = False
    for url in MIRRORS:
        try:
            print(f"Trying: {url}")
            with requests.get(url, stream=True, timeout=30) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                done  = 0
                with open(KAFKA_TGZ, "wb") as f:
                    for chunk in r.iter_content(chunk_size=65536):
                        f.write(chunk)
                        done += len(chunk)
                        if total:
                            print(f"\r  {done/1e6:.1f} / {total/1e6:.1f} MB", end="", flush=True)
            print("\nDownload complete.")
            downloaded = True
            break
        except Exception as e:
            print(f"\nFailed ({url}): {e}")
    if not downloaded:
        raise RuntimeError("All mirrors failed. Check your internet connection.")
    print("Extracting...")
    with tarfile.open(KAFKA_TGZ, "r:gz") as tar:
        tar.extractall()
    print("Extracted.")
else:
    print("Kafka already extracted, skipping download.")

KAFKA_HOME = os.path.abspath(KAFKA_DIR)

# ---------- Windows-safe paths ----------
TEMP_DIR     = tempfile.gettempdir()
zk_log_path  = os.path.join(TEMP_DIR, "zookeeper.log")
kb_log_path  = os.path.join(TEMP_DIR, "kafka.log")
zk_log       = open(zk_log_path, "w")
kb_log       = open(kb_log_path, "w")

# ---------- Build a SHORT wildcard classpath (avoids "input line too long") ----------
libs_dir   = os.path.join(KAFKA_HOME, "libs")
classpath  = os.path.join(libs_dir, "*")
log4j_conf = os.path.join(KAFKA_HOME, "config", "log4j.properties")

base_java_opts = [
    "java",
    "-cp", classpath,
    f"-Dlog4j.configuration=file:{log4j_conf}",
]

# ---------- Start ZooKeeper directly via Java (bypassing zookeeper-server-start.bat) ----------
zk_proc = subprocess.Popen(
    base_java_opts + [
        "org.apache.zookeeper.server.quorum.QuorumPeerMain",
        os.path.join(KAFKA_HOME, "config", "zookeeper.properties"),
    ],
    stdout=zk_log, stderr=zk_log,
    cwd=KAFKA_HOME,
)
print("ZooKeeper starting...")
time.sleep(8)

# ---------- Start Kafka broker directly via Java (bypassing kafka-server-start.bat) ----------
kb_proc = subprocess.Popen(
    base_java_opts + [
        "kafka.Kafka",
        os.path.join(KAFKA_HOME, "config", "server.properties"),
    ],
    stdout=kb_log, stderr=kb_log,
    cwd=KAFKA_HOME,
)
print("Kafka broker starting...")
time.sleep(20)

# ---------- Verify with kafka-topics, also via direct Java call ----------
result = subprocess.run(
    base_java_opts + [
        "kafka.admin.TopicCommand",
        "--bootstrap-server", "localhost:9092",
        "--list",
    ],
    capture_output=True, text=True, timeout=15,
    cwd=KAFKA_HOME,
)

if result.returncode == 0:
    print("\n✔ Kafka is ready! Topics:", result.stdout.strip() or "(none yet)")
else:
    print("✘ Kafka did not start. Last 20 lines of kafka.log:")
    with open(kb_log_path) as f:
        print("".join(f.readlines()[-20:]))
    print("\nLast 20 lines of zookeeper.log:")
    with open(zk_log_path) as f:
        print("".join(f.readlines()[-20:]))

Kafka already extracted, skipping download.
ZooKeeper starting...
Kafka broker starting...

✔ Kafka is ready! Topics: [2026-06-17 23:21:40,084] INFO Registered kafka:type=kafka.Log4jController MBean (kafka.utils.Log4jControllerRegistration$)
[2026-06-17 23:21:40,564] INFO AdminClientConfig values: 
	auto.include.jmx.reporter = true
	bootstrap.servers = [localhost:9092]
	client.dns.lookup = use_all_dns_ips
	client.id = 
	connections.max.idle.ms = 300000
	default.api.timeout.ms = 60000
	metadata.max.age.ms = 300000
	metric.reporters = []
	metrics.num.samples = 2
	metrics.recording.level = INFO
	metrics.sample.window.ms = 30000
	receive.buffer.bytes = 65536
	reconnect.backoff.max.ms = 1000
	reconnect.backoff.ms = 50
	request.timeout.ms = 30000
	retries = 2147483647
	retry.backoff.ms = 100
	sasl.client.callback.handler.class = null
	sasl.jaas.config = null
	sasl.kerberos.kinit.cmd = /usr/bin/kinit
	sasl.kerberos.min.time.before.relogin = 60000
	sasl.kerberos.service.name = null
	sasl.kerbe

## Step 13b — Kafka Producer
This defines the producer and `send_articles_to_kafka` function.
Sends cleaned news articles to the Kafka topic `news_raw`.

In [16]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)
print("KafkaProducer connected.")

def send_articles_to_kafka(articles, topic="news_raw"):
    """Send a list of article dicts to the given Kafka topic."""
    sent = 0
    for article in articles:
        producer.send(topic, article)
        sent += 1
    producer.flush()
    print(f"Sent {sent} articles to Kafka topic '{topic}'.")

KafkaProducer connected.


## Step 14 — Publish Week 1 Articles to Kafka

In [17]:


if 'all_articles' in globals() and all_articles:
    send_articles_to_kafka(all_articles)
else:
    print("No articles found. Run the Week 1 pipeline first to create `all_articles`.")

Sent 51 articles to Kafka topic 'news_raw'.


## Step 15 — Kafka Consumer (Verification)
Reads a few messages from Kafka to confirm data is arriving.

In [18]:
from kafka import KafkaConsumer

def preview_kafka_messages(topic="news_raw", max_messages=5):
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers="localhost:9092",
        auto_offset_reset="earliest",
        enable_auto_commit=True,
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=5000
    )

    count = 0
    for message in consumer:
        print(json.dumps(message.value, indent=2)[:1000])
        print("-" * 80)
        count += 1
        if count >= max_messages:
            break

    consumer.close()
    print(f"Previewed {count} messages.")

preview_kafka_messages()

C:\Program Files\KMSpico\temp\ipykernel_17968\2637545772.py:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


{
  "title": "Shaftesbury Theatre in West End to be named after Dame Judi Dench",
  "source": "NewsAPI",
  "description": "The Shaftesbury Theatre is to be renamed to honour Dame Judis contribution to the performing arts",
  "content": "The Shaftesbury Theatre in Londons West End is to be renamed after the actress Dame Judi Dench to celebrate her unparalleled contribution to British theatre and the performing arts The renaming 1299 chars",
  "url": "https://www.bbc.co.uk/news/articles/c4gy4d0d7qjo",
  "published_at": "2026-06-16T15:52:22.7553145Z"
}
--------------------------------------------------------------------------------
{
  "title": "MoD investigating reports Russian warship fired warning shots near yacht in Channel",
  "source": "NewsAPI",
  "description": "MoD investigating reports Russian warship fired warning shots near yacht in Channel",
  "content": "The Ministry of Defence is investigating reports a Russian warship fired warning shots near a UKregistered yacht in the En

## Step 16 — Structured Streaming Schema

In [19]:
from pyspark.sql.types import StructType, StructField, StringType

news_schema = StructType([
    StructField("source",       StringType(), True),
    StructField("title",        StringType(), True),
    StructField("description",  StringType(), True),
    StructField("content",      StringType(), True),
    StructField("url",          StringType(), True),
    StructField("published_at", StringType(), True),
])

## Step 17 — Read Kafka Stream with PySpark

In [20]:
print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionString())

Spark: 4.1.2
Scala: version 2.13.17


In [21]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "news_raw")
    .option("startingOffsets", "earliest")
    .load()
)

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Step 18 — Parse JSON and Clean Data

In [22]:
from pyspark.sql.functions import col, from_json, lower, trim
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

parsed_stream = (
    raw_stream
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), news_schema).alias("data"))
    .select("data.*")
)

clean_stream = (
    parsed_stream
    .withColumn("title",       trim(col("title")))
    .withColumn("description", trim(col("description")))
    .withColumn("source",      lower(trim(col("source"))))
    .dropDuplicates(["url"])
    .filter(col("title").isNotNull())
)

clean_stream.printSchema()

root
 |-- source: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- content: string (nullable = true)
 |-- url: string (nullable = true)
 |-- published_at: string (nullable = true)



## Step 19 — Start Streaming Query

In [23]:
query = (
    clean_stream.writeStream
    .format("memory")
    .queryName("clean_news")
    .outputMode("append")
    .start()
)

print("Streaming query started.")

Streaming query started.


In [33]:
query = clean_stream.writeStream \
    .format("memory") \
    .queryName("clean_news") \
    .outputMode("append") \
    .start()

print("quret is active ",query.isActive)

quret is active  True


In [34]:
query.lastProgress

In [35]:
query = clean_stream.writeStream \
    .format("memory") \
    .queryName("clean_news") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .start()

## Step 20 — View Processed News

In [36]:
while query.isActive:
    spark.sql("SELECT * FROM clean_news").show()
    time.sleep(5)

In [37]:
import time
time.sleep(20)

spark.sql("""
SELECT source, title, published_at
FROM clean_news
LIMIT 20
""").show(truncate=False)

+------+-----+------------+
|source|title|published_at|
+------+-----+------------+
+------+-----+------------+



## Step 21 — Save Processed Data to JSON

In [ ]:
import requests, json
from bs4 import BeautifulSoup

# Load from Spark BEFORE stopping it
processed_df = spark.sql("SELECT * FROM clean_news")
articles = processed_df.toPandas().to_dict(orient="records")
print(f"Got {len(articles)} articles from Spark")

def fetch_article_content(url: str) -> str:
    try:
        resp = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(resp.text, "html.parser")
        paragraphs = soup.find_all("p")
        text = " ".join(p.get_text() for p in paragraphs)
        return text[:2000]
    except Exception:
        return ""

for a in articles:
    if not a.get("content") and not a.get("description"):
        print(f"Fetching: {a['title'][:60]}")
        a["content"] = fetch_article_content(a["url"])

def is_english(text):
    if not text: return False
    return sum(1 for c in text if ord(c) < 128) / len(text) > 0.8

articles = [a for a in articles if is_english(a.get("title", ""))]

with open("week2_processed_news.json", "w", encoding="utf-8") as f:
    json.dump(articles, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(articles)} clean articles to week2_processed_news.json")
print(f"Articles with content: {sum(1 for a in articles if a.get('content'))}/{len(articles)}")

In [ ]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-community \
             transformers torch nltk textblob tqdm
print("Week 3 dependencies installed.")


In [ ]:
import json

with open("week2_processed_news.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

print(f"Loaded {len(articles)} articles from week2_processed_news.json")
print(f"Columns: {list(articles[0].keys())}")
print(f"\nSample article:")
print(json.dumps(articles[0], indent=2))


In [ ]:
for a in articles:
    title   = a.get("title", "") or ""
    desc    = a.get("description", "") or ""
    content = a.get("content", "") or ""
    a["full_text"] = f"{title}. {desc} {content}".strip()

# Verify
non_empty = sum(1 for a in articles if len(a["full_text"]) > 10)
print(f"{non_empty}/{len(articles)} articles have non-empty full_text")
print(f"\nExample full_text:\n{articles[0]['full_text'][:300]}")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

texts = [a["full_text"] for a in articles]

print(f"Encoding {len(texts)} articles...")
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"Each article → {embeddings.shape[1]}-dim vector")


In [ ]:
import faiss, json, os

dimension = embeddings.shape[1]   # 384

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"FAISS index built — {index.ntotal} vectors, dim={dimension}")

# Persist
faiss.write_index(index, "news_faiss.index")

metadata = [
    {"title": a["title"], "source": a["source"],
     "url": a["url"], "published_at": a["published_at"],
     "full_text": a["full_text"]}
    for a in articles
]
with open("news_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved → news_faiss.index")
print("Saved → news_metadata.json")


In [ ]:
def semantic_search(query: str, top_k: int = 3) -> list:
    """
    Embed `query` and return the top_k most relevant articles from FAISS.
    """
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        entry = dict(metadata[idx])
        entry["score"] = round(float(score), 4)
        results.append(entry)
    return results


test_results = semantic_search("technology and artificial intelligence", top_k=3)
print("Semantic search test:")
for r in test_results:
    print(f"  [{r['score']:.4f}] {r['title']} ({r['source']})")


In [ ]:
import os

!pip install -q groq

from groq import Groq

GROQ_API_KEY = "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"  # ← paste your key from console.groq.com

try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

GROQ_API_KEY = GROQ_API_KEY or os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY is not set. Get a free key at console.groq.com")

groq_client = Groq(api_key=GROQ_API_KEY)

def _call_llm(prompt: str, max_tokens: int = 512) -> str:
    """
    Call Groq (Llama 3) with the given prompt.
    Drop-in replacement — same signature as before.
    """
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


print("LLM helper ready. Key configured:", bool(GROQ_API_KEY))

In [ ]:
def rag_answer(user_query: str, top_k: int = 3) -> str:
    """
    Full RAG pipeline:
      1. Retrieve top_k articles from FAISS.
      2. Build a grounded prompt.
      3. Call Claude and return the answer.
    """
    retrieved = semantic_search(user_query, top_k=top_k)

    context_blocks = []
    for i, art in enumerate(retrieved, 1):
        context_blocks.append(
            f"Article {i} (source: {art['source']}, score: {art['score']}):\n"
            f"Title: {art['title']}\n"
            f"Text: {art['full_text']}"
        )
    context = "\n\n".join(context_blocks)

    prompt = (
        "You are a news intelligence assistant. "
        "Answer the question ONLY using the provided news articles. "
        "Do not fabricate information not present in the articles.\n\n"
        f"News Articles:\n{context}\n\n"
        f"Question: {user_query}\n\nAnswer:"
    )
    return _call_llm(prompt)



answer = rag_answer("What are the latest technology developments?")
print(answer)


In [ ]:
import re as _re

def _parse_confidence(value) -> float:
    """
    Robustly parse a confidence value from LLM output.
    Handles: '0.8', '~0.7', '0.8-0.9', '85%', 'high', etc.
    """
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip()
    match = _re.search(r'\d+\.?\d*', s)
    if not match:
        return 0.5
    num = float(match.group())
    if num > 1.0:
        num = num / 100.0
    return round(min(max(num, 0.0), 1.0), 4)


def sentiment_agent(article: dict) -> dict:
    """
    Classifies the sentiment of a single article.
    Returns: {title, sentiment, confidence, reason}
    """
    text = article.get("full_text", "")
    prompt = (
        f"Classify the sentiment of this news article as Positive, Negative, or Neutral.\n"
        f"Article: {text[:800]}\n\n"
        "Reply in EXACTLY this format with no extra text:\n"
        "Sentiment: Positive\n"
        "Confidence: 0.85\n"
        "Reason: One sentence explanation.\n\n"
        "Use only a plain decimal number (e.g. 0.85) for Confidence."
    )
    raw = _call_llm(prompt, max_tokens=120)
    lines = {l.split(":")[0].strip(): ":".join(l.split(":")[1:]).strip()
             for l in raw.splitlines() if ":" in l}

    sentiment_raw = lines.get("Sentiment", "Neutral").strip().strip("*_")
    sentiment = "Neutral"
    for label in ("Positive", "Negative", "Neutral"):
        if label.lower() in sentiment_raw.lower():
            sentiment = label
            break

    return {
        "title":      article["title"],
        "sentiment":  sentiment,
        "confidence": _parse_confidence(lines.get("Confidence", 0.5)),
        "reason":     lines.get("Reason", ""),
    }


print("Running Sentiment Analysis Agent on all articles...\n")
sentiment_results = [sentiment_agent(a) for a in articles]

from collections import Counter
for r in sentiment_results:
    icon = {"Positive": "🟢", "Negative": "🔴", "Neutral": "⚪"}.get(r["sentiment"], "")
    print(f"{icon} [{r['sentiment']:8s}] ({r['confidence']:.2f}) {r['title']}")

counts = Counter(r["sentiment"] for r in sentiment_results)
print(f"\nDistribution: {dict(counts)}")


In [ ]:
def summarization_agent(article: dict) -> dict:
    """
    Summarizes a single article in 2-3 sentences.
    Returns: {title, summary}
    """
    text = article.get("full_text", "")
    prompt = (
        "Summarize the following news article in exactly 2-3 sentences. "
        "Be factual and concise.\n\n"
        f"Article: {text[:1200]}\n\nSummary:"
    )
    return {"title": article["title"], "summary": _call_llm(prompt, max_tokens=150).strip()}


print("Running Summarization Agent...\n")
summaries = [summarization_agent(a) for a in articles]

for i, s in enumerate(summaries, 1):
    print(f"{i}. {s['title']}")
    print(f"   {s['summary']}")
    print()


In [ ]:
import nltk
from collections import Counter

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOPWORDS = set(stopwords.words("english"))


def _extract_keywords(text: str, top_n: int = 10) -> list:
    tokens = word_tokenize(text.lower())
    tagged = nltk.pos_tag(tokens)
    keywords = [
        word for word, pos in tagged
        if pos.startswith(("NN", "NNP"))
        and word.isalpha()
        and word not in STOPWORDS
        and len(word) > 3
    ]
    return [w for w, _ in Counter(keywords).most_common(top_n)]


def trend_detection_agent(article_list: list, top_topics: int = 10) -> dict:
    """
    Extracts top keywords from all articles and generates a trend report.
    """
    all_text = " ".join(a["full_text"] for a in article_list)
    top_keywords = _extract_keywords(all_text, top_n=top_topics)

    prompt = (
        f"Based on these trending keywords from today's live news stream: {top_keywords}\n"
        "Write a short 3-4 sentence trend report identifying the main themes and what they suggest."
    )
    report = _call_llm(prompt, max_tokens=200).strip()

    return {"top_keywords": top_keywords, "trend_report": report}


print("Running Trend Detection Agent...\n")
trends = trend_detection_agent(articles)

print("Top Trending Keywords:")
print(" | ".join(trends["top_keywords"]))
print("\nTrend Report:")
print(trends["trend_report"])


In [ ]:
def qa_agent(question: str, top_k: int = 3) -> dict:
    """
    Q&A Agent — retrieves relevant articles and answers the question.
    Returns: {question, answer, sources}
    """
    retrieved = semantic_search(question, top_k=top_k)

    context_blocks = []
    for i, art in enumerate(retrieved, 1):
        context_blocks.append(
            f"[Article {i}] Title: {art['title']}\n"
            f"Source: {art['source']} | Published: {art['published_at']}\n"
            f"Text: {art['full_text'][:600]}"
        )
    context = "\n\n".join(context_blocks)

    prompt = (
        "You are a news Q&A assistant. Answer the question based ONLY on the "
        "provided articles. If the answer is not in the articles, say so.\n\n"
        f"{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer = _call_llm(prompt, max_tokens=300).strip()
    sources = [{"title": r["title"], "source": r["source"],
                "url": r["url"], "score": r["score"]} for r in retrieved]

    return {"question": question, "answer": answer, "sources": sources}


# Demo
result = qa_agent("What are the most important news stories right now?")
print(f"Q: {result['question']}")
print(f"\nA: {result['answer']}")
print("\nSources used:")
for s in result["sources"]:
    print(f"  [{s['score']:.3f}] {s['title']} ({s['source']})")


In [ ]:

from langgraph.graph import StateGraph, END
from typing import TypedDict

class NewsAgentState(TypedDict):
    query:               str
    mode:                str
    retrieved_articles:  list
    results:             dict

def router_node(state: NewsAgentState) -> NewsAgentState:
    mode = state["mode"]
    if mode == "auto":
        q = state["query"].lower()
        if any(w in q for w in ["summarize", "summary", "brief", "overview"]):
            mode = "summary"
        elif any(w in q for w in ["sentiment", "positive", "negative", "tone", "feeling"]):
            mode = "sentiment"
        elif any(w in q for w in ["trend", "trending", "topic", "theme"]):
            mode = "trends"
        else:
            mode = "qa"
    return {**state, "mode": mode}

def retrieval_node(state: NewsAgentState) -> NewsAgentState:
    retrieved = [
        next((a for a in articles if a["title"] == r["title"]), r)
        for r in semantic_search(state["query"], top_k=3)
    ]
    return {**state, "retrieved_articles": retrieved}

def qa_node(state: NewsAgentState) -> NewsAgentState:
    result = qa_agent(state["query"])
    return {**state, "results": {**state["results"], "qa": result}}

def summary_node(state: NewsAgentState) -> NewsAgentState:
    summaries = [summarization_agent(a) for a in state["retrieved_articles"]]
    return {**state, "results": {**state["results"], "summaries": summaries}}

def sentiment_node(state: NewsAgentState) -> NewsAgentState:
    sentiment = [sentiment_agent(a) for a in state["retrieved_articles"]]
    return {**state, "results": {**state["results"], "sentiment": sentiment}}

def trends_node(state: NewsAgentState) -> NewsAgentState:
    trends = trend_detection_agent(articles)
    return {**state, "results": {**state["results"], "trends": trends}}

def full_node(state: NewsAgentState) -> NewsAgentState:
    state = qa_node(state)
    state = summary_node(state)
    state = sentiment_node(state)
    state = trends_node(state)
    return state

def route_after_retrieval(state: NewsAgentState) -> str:
    return state["mode"]

workflow = StateGraph(NewsAgentState)

workflow.add_node("router",    router_node)
workflow.add_node("retrieval", retrieval_node)
workflow.add_node("qa",        qa_node)
workflow.add_node("summary",   summary_node)
workflow.add_node("sentiment", sentiment_node)
workflow.add_node("trends",    trends_node)
workflow.add_node("full",      full_node)

workflow.set_entry_point("router")
workflow.add_edge("router", "retrieval")

workflow.add_conditional_edges(
    "retrieval",
    route_after_retrieval,
    {
        "qa":        "qa",
        "summary":   "summary",
        "sentiment": "sentiment",
        "trends":    "trends",
        "full":      "full",
    }
)

workflow.add_edge("qa",        END)
workflow.add_edge("summary",   END)
workflow.add_edge("sentiment", END)
workflow.add_edge("trends",    END)
workflow.add_edge("full",      END)

news_graph = workflow.compile()

print("✔ LangGraph news pipeline compiled successfully.")
print("  Nodes:", list(news_graph.get_graph().nodes.keys()))

def multi_agent_pipeline(user_input: str, mode: str = "auto") -> dict:
    """
    LangGraph-powered Multi-Agent Orchestrator.

    mode='auto'      → detect intent, route to SINGLE best agent
    mode='full'      → run ALL four agents
    mode='qa'        → Q&A Agent only
    mode='sentiment' → Sentiment Agent only
    mode='summary'   → Summarization Agent only
    mode='trends'    → Trend Detection Agent only
    """
    initial_state: NewsAgentState = {
        "query":              user_input,
        "mode":               mode,
        "retrieved_articles": [],
        "results":            {},
    }
    final_state = news_graph.invoke(initial_state)
    return {
        "query":   user_input,
        "mode":    final_state["mode"],
        "results": final_state["results"],
    }


print("\n" + "=" * 60)
print(" Multi-Agent Pipeline (LangGraph) — Demo")
print("=" * 60)

result = multi_agent_pipeline("What is happening in the news today?", mode="auto")
print(f"\n[auto mode resolved to: {result['mode']}]")
for key, val in result["results"].items():
    print(f"  Agent returned: {key}")

result_full = multi_agent_pipeline("Give me a full briefing", mode="full")
print(f"\n[full mode] Agents ran: {list(result_full['results'].keys())}")

print("\n" + "=" * 60)


In [ ]:
import json, os

week3_output = {
    "total_articles_processed": len(articles),
    "sentiment_analysis": sentiment_results,
    "summaries":          summaries,
    "trends":             trends,
}

with open("week3_results.json", "w", encoding="utf-8") as f:
    json.dump(week3_output, f, indent=2, ensure_ascii=False)

print("Saved → week3_results.json")
for fname in ["news_faiss.index", "news_metadata.json", "week3_results.json"]:
    print(f"  {'✔' if os.path.exists(fname) else '✘'}  {fname}")


In [ ]:
# Run when you are finished.
query.stop()
spark.stop()

# Week 4 — Frontend, Testing & Finalization


## Step 22 — Install Week 4 Dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] streamlit pyngrok nest_asyncio httpx plotly langgraph
print('Week 4 dependencies installed.')

In [ ]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

import uvicorn, threading, json, os
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Real-Time News Intelligence Agent", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class QueryRequest(BaseModel):
    query: str
    mode: Optional[str] = "auto"


@app.get("/health")
def health():
    return {"status": "ok", "articles_loaded": len(articles)}


@app.post("/query")
def query_endpoint(req: QueryRequest):
    try:
        result = multi_agent_pipeline(req.query, mode=req.mode)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/sentiment")
def sentiment_endpoint():
    """
    Returns sentiment distribution across all articles.
    """
    from collections import Counter
    results = [sentiment_agent(a) for a in articles]
    counts  = Counter(r["sentiment"] for r in results)
    return {
        "distribution": dict(counts),
        "details": results,
    }


@app.get("/trends")
def trends_endpoint():
    """
    Returns trending keywords and LLM trend report.
    """
    return trend_detection_agent(articles)


@app.get("/articles")
def articles_endpoint(page: int = 1, page_size: int = 10):
    """
    Returns a paginated list of processed articles.
    """
    start = (page - 1) * page_size
    end   = start + page_size
    return {
        "total":    len(articles),
        "page":     page,
        "articles": articles[start:end],
    }


def run_api():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    asyncio.run(server.serve())

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

import time; time.sleep(5)
print("FastAPI backend running on http://localhost:8000")
print("Docs available at  http://localhost:8000/docs")

## Step 24 — Verify API Endpoints
Quick sanity checks using `httpx` to confirm all four endpoints respond correctly.

In [ ]:
import httpx, json

BASE = "http://localhost:8000"

r = httpx.get(f"{BASE}/health")
print("Health:", r.json())

r = httpx.post(f"{BASE}/query",
               json={"query": "What is happening in AI?", "mode": "qa"},
               timeout=60)
print("\nQuery (QA mode):")
resp = r.json()
if "results" in resp and "qa" in resp["results"]:
    print(resp["results"]["qa"]["answer"][:300])

r = httpx.get(f"{BASE}/trends", timeout=60)
trends_data = r.json()
print("\nTrends keywords:", trends_data.get("top_keywords"))

r = httpx.get(f"{BASE}/articles?page=1&page_size=3")
art_data = r.json()
print(f"\nArticles (total={art_data['total']}), showing page 1:")
for a in art_data["articles"]:
    print(f"  [{a['source']}] {a['title']}")


## Step 25 — Streamlit Chatbot UI & Analytics Dashboard
### `frontend/app.py`

Writes the Streamlit application to disk. The app has two tabs:

- **💬 Chat** — natural language Q&A against the live news stream
- **📊 Analytics** — sentiment pie-chart, trending keywords bar-chart, and article browser

In [ ]:
streamlit_code = '''
import streamlit as st
import requests, json, plotly.express as px, pandas as pd

API_BASE = "http://localhost:8000"

st.set_page_config(
    page_title="Real-Time News Intelligence Agent",
    page_icon="📰",
    layout="wide",
)

st.title("📰 Real-Time News Intelligence Agent")
st.caption("Powered by Apache Kafka · PySpark · FAISS · LangChain · Groq LLaMA-3")

tab_chat, tab_analytics = st.tabs(["💬 Chat", "📊 Analytics"])

# ── Chat Tab ──────────────────────────────────────────────────────────────────
with tab_chat:
    st.subheader("Ask anything about today's news")

    if "history" not in st.session_state:
        st.session_state.history = []

    mode = st.selectbox(
        "Agent mode",
        ["qa", "summary", "full"],
        help="auto lets the system pick the best agent based on your query"
    )

    with st.form("chat_form", clear_on_submit=True):
        user_input = st.text_input("Your question", placeholder="What is happening in AI today?")
        submitted  = st.form_submit_button("Send")

    if submitted and user_input.strip():
        with st.spinner("Thinking..."):
            try:
                resp = requests.post(
                    f"{API_BASE}/query",
                    json={"query": user_input, "mode": mode},
                    timeout=90
                )
                data = resp.json()
                st.session_state.history.append({"role": "user",      "content": user_input})
                st.session_state.history.append({"role": "assistant", "content": data})
            except Exception as e:
                st.error(f"API error: {e}")

    for msg in reversed(st.session_state.history):
        if msg["role"] == "user":
            st.chat_message("user").write(msg["content"])
        else:
            data = msg["content"]
            with st.chat_message("assistant"):
                results = data.get("results", {})
                mode = data.get("mode", "auto")

                # qa mode (or auto resolved to qa)
                if "qa" in results:
                    st.markdown(results["qa"]["answer"])
                    with st.expander("Sources"):
                        for s in results["qa"].get("sources", []):
                            st.write(f"[{s[\"score\"]:.3f}] [{s[\"title\"]}]({s[\"url\"]}) — {s[\"source\"]}")

                # summary mode
                if "summaries" in results:
                    if mode in ("summary", "full"):
                        st.markdown("**Summaries:**")
                        for s in results["summaries"]:
                            st.write(f"• **{s[\"title\"]}** — {s[\"summary\"]}")

                # sentiment mode
                if "sentiment" in results:
                    if mode in ("sentiment", "full"):
                        st.markdown("**Sentiment:**")
                        for s in results["sentiment"]:
                            icon = {"Positive": "🟢", "Negative": "🔴", "Neutral": "⚪"}.get(s[\"sentiment\"], "")
                            st.write(f"{icon} {s[\"sentiment\"]} ({s[\"confidence\"]:.2f}) — {s[\"title\"]}")

                # trends mode
                if "trends" in results:
                    if mode in ("trends", "full"):
                        st.markdown("**Trends:**")
                        st.write(" | ".join(results["trends"]["top_keywords"]))
                        st.write(results["trends"]["trend_report"])

# ── Analytics Tab ─────────────────────────────────────────────────────────────
with tab_analytics:
    st.subheader("Live Analytics Dashboard")

    col1, col2 = st.columns(2)

    # Sentiment pie chart
    with col1:
        st.markdown("### Sentiment Distribution")
        if st.button("Load Sentiment"):
            with st.spinner("Running sentiment analysis..."):
                try:
                    r = requests.get(f"{API_BASE}/sentiment", timeout=120)
                    dist = r.json()["distribution"]
                    df_pie = pd.DataFrame(list(dist.items()), columns=["Sentiment", "Count"])
                    fig = px.pie(df_pie, names="Sentiment", values="Count",
                                 color="Sentiment",
                                 color_discrete_map={"Positive": "green", "Negative": "red", "Neutral": "gray"},
                                 title="Sentiment Breakdown")
                    st.plotly_chart(fig, use_container_width=True)
                except Exception as e:
                    st.error(f"Error: {e}")

    # Trending keywords bar chart
    with col2:
        st.markdown("### Trending Keywords")
        if st.button("Load Trends"):
            with st.spinner("Extracting trends..."):
                try:
                    r = requests.get(f"{API_BASE}/trends", timeout=60)
                    data = r.json()
                    keywords = data["top_keywords"]
                    df_kw = pd.DataFrame({"Keyword": keywords, "Rank": range(1, len(keywords)+1)})
                    fig = px.bar(df_kw, x="Keyword", y="Rank", title="Top Trending Keywords",
                                 color="Rank", color_continuous_scale="blues")
                    fig.update_yaxes(autorange="reversed")
                    st.plotly_chart(fig, use_container_width=True)
                    st.info(data["trend_report"])
                except Exception as e:
                    st.error(f"Error: {e}")

    # Article browser
    st.markdown("### Article Browser")
    page = st.number_input("Page", min_value=1, value=1, step=1)
    if st.button("Load Articles"):
        try:
            r = requests.get(f"{API_BASE}/articles", params={"page": page, "page_size": 10})
            art_data = r.json()
            st.write(f"Total articles: **{art_data[\"total\"]}**")
            df = pd.DataFrame(art_data["articles"])[["source", "title", "published_at", "url"]]
            st.dataframe(df, use_container_width=True)
        except Exception as e:
            st.error(f"Error: {e}")
'''

import os
os.makedirs("frontend", exist_ok=True)
with open("frontend/app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("Streamlit app written to frontend/app.py")
print("Run it (outside Colab) with:  streamlit run frontend/app.py")


## Step 26 — Launch Streamlit in Colab (with pyngrok)

> **Note:** pyngrok tunnels the Streamlit UI to a public URL inside Colab.
> If running locally, skip this cell and simply run `streamlit run frontend/app.py` in your terminal.

In [ ]:
from pyngrok import ngrok
import subprocess, time, sys

NGROK_AUTH_TOKEN = "3EFgZVgiPMr4YEkNaa7kxs1pTFM_7P2sRnQHcSzhbA5uLiLHN"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "frontend/app.py",
        "--server.port",       "8501",
        "--server.headless",   "true",
        "--server.enableCORS", "false",
        "--server.address",    "0.0.0.0",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("Waiting for Streamlit to start...", end="", flush=True)
for _ in range(12):
    time.sleep(1)
    print(".", end="", flush=True)
print(" done.")

if streamlit_proc.poll() is not None:
    err = streamlit_proc.stderr.read().decode()
    raise RuntimeError(f"Streamlit failed to start:\n{err}")

ngrok.kill()
time.sleep(1)

tunnel     = ngrok.connect(8501, "http")
public_url = tunnel.public_url
print(f"\n🌐 Streamlit UI  →  {public_url}")
print("📖 FastAPI Docs  →  http://localhost:8000/docs")
print("\nOpen the Streamlit URL above in your browser.")
print("The link stays active as long as this Colab session is running.")


## Step 27 — End-to-End Evaluation

Runs the five manual evaluation checks defined in the project proposal (Section 10) and records pass/fail with evidence.

In [ ]:
import time, json
from collections import Counter

print("=" * 60)
print(" Real-Time News Intelligence Agent — Week 4 Evaluation")
print("=" * 60)

eval_results = {}

print("\n[Eval 1/5] Chatbot Relevance — 10 diverse queries")
test_queries = [
    "What is happening in artificial intelligence?",
    "Tell me about recent political events.",
    "What are the latest technology developments?",
    "Give me news about the economy and finance.",
    "What is going on in sports?",
    "Are there any climate or environmental news?",
    "What is the latest in health and medicine?",
    "Tell me about international relations news.",
    "What happened in the business world recently?",
    "Are there any breaking news stories?",
]

relevance_pass = 0
for i, q in enumerate(test_queries, 1):
    result = qa_agent(q, top_k=3)
    answer = result["answer"]
    passed = len(answer) > 30 and "cannot" not in answer.lower()[:60]
    icon   = "✔" if passed else "✘"
    if passed:
        relevance_pass += 1
    print(f"  {icon} Q{i}: {q[:55]}")

eval_results["relevance"] = {"pass": relevance_pass, "total": 10}
print(f"  → {relevance_pass}/10 queries returned relevant answers.")


print("\n[Eval 2/5] Summary Quality — 5 articles")
sample = articles[:5]
summary_pass = 0
for a in sample:
    s = summarization_agent(a)
    passed = 40 <= len(s["summary"]) <= 500
    icon   = "✔" if passed else "✘"
    if passed:
        summary_pass += 1
    print(f"  {icon} {a['title'][:60]}")
    print(f"     → {s['summary'][:120]}")

eval_results["summary_quality"] = {"pass": summary_pass, "total": 5}
print(f"  → {summary_pass}/5 summaries passed quality check.")


print("\n[Eval 3/5] Sentiment Accuracy — labelled test set")
sentiment_test_set = [
    {"full_text": "Scientists celebrate major breakthrough in cancer treatment, offering new hope to millions of patients worldwide.",   "title": "Cancer Breakthrough",   "expected": "Positive"},
    {"full_text": "Stock markets crash as economic recession fears grow, wiping billions off global indices in hours.",                  "title": "Market Crash",          "expected": "Negative"},
    {"full_text": "The government released its annual budget report outlining planned infrastructure spending for the coming year.",    "title": "Budget Report",         "expected": "Neutral"},
    {"full_text": "Record rainfall brings devastating floods, destroying thousands of homes and displacing entire communities.",        "title": "Devastating Floods",    "expected": "Negative"},
    {"full_text": "Olympic athletes break three world records in a historic games celebrated by fans around the globe.",               "title": "Olympic Records",       "expected": "Positive"},
]

sentiment_pass = 0
for item in sentiment_test_set:
    pred = sentiment_agent(item)
    passed = pred["sentiment"].lower() == item["expected"].lower()
    icon   = "✔" if passed else "✘"
    if passed:
        sentiment_pass += 1
    print(f"  {icon} Expected={item['expected']:8s} Got={pred['sentiment']:8s} — {item['title']}")

eval_results["sentiment_accuracy"] = {"pass": sentiment_pass, "total": 5}
print(f"  → {sentiment_pass}/5 sentiment labels correct.")


print("\n[Eval 4/5] Latency — 3 end-to-end pipeline calls (target < 30 s)")
latency_pass = 0
latencies    = []
for i in range(3):
    start = time.time()
    _     = multi_agent_pipeline("What is the top news story today?", mode="qa")
    elapsed = time.time() - start
    latencies.append(elapsed)
    passed = elapsed < 30
    if passed:
        latency_pass += 1
    print(f"  {'✔' if passed else '✘'} Run {i+1}: {elapsed:.2f}s")

avg_latency = sum(latencies) / len(latencies)
eval_results["latency"] = {"pass": latency_pass, "total": 3, "avg_seconds": round(avg_latency, 2)}
print(f"  → Average latency: {avg_latency:.2f}s  |  {latency_pass}/3 under 30 s threshold.")


print("\n[Eval 5/5] Pipeline Stability — 10 consecutive multi-agent calls")
stability_pass = 0
for i in range(10):
    try:
        r = multi_agent_pipeline(test_queries[i % len(test_queries)], mode="qa")
        assert "results" in r
        stability_pass += 1
        print(f"  ✔ Call {i+1:2d} succeeded")
    except Exception as e:
        print(f"  ✘ Call {i+1:2d} FAILED: {e}")

eval_results["stability"] = {"pass": stability_pass, "total": 10}
print(f"  → {stability_pass}/10 calls completed without errors.")


print("\n" + "=" * 60)
print(" EVALUATION SUMMARY")
print("=" * 60)
labels = {
    "relevance":           "Chatbot Relevance",
    "summary_quality":     "Summary Quality",
    "sentiment_accuracy":  "Sentiment Accuracy",
    "latency":             "Latency (< 30 s)",
    "stability":           "Pipeline Stability",
}
all_pass = True
for key, label in labels.items():
    r  = eval_results[key]
    ok = r["pass"] == r["total"]
    if not ok:
        all_pass = False
    icon = "✔" if ok else "△"
    extra = f"  avg={r['avg_seconds']}s" if "avg_seconds" in r else ""
    print(f"  {icon}  {label}: {r['pass']}/{r['total']}{extra}")

print("=" * 60)
print("Overall:", "ALL CHECKS PASSED ✔" if all_pass else "Some checks need attention — see above.")

eval_results["summary_overall_pass"] = all_pass

with open("week4_eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Saved → week4_eval_results.json")


## Step 28 — Generate Final Report
Produces a structured Markdown report documenting architecture, evaluation results, learnings, and future work.

In [ ]:

import os, json

checks = {
    "FastAPI backend running":         True,  # confirmed in Step 23
    "frontend/app.py (Streamlit UI)": os.path.exists("frontend/app.py"),
    "week4_eval_results.json exists":  os.path.exists("week4_eval_results.json"),
    "final_report.md exists":          os.path.exists("final_report.md"),
    "news_faiss.index exists":         os.path.exists("news_faiss.index"),
    "news_metadata.json exists":       os.path.exists("news_metadata.json"),
    "week3_results.json exists":       os.path.exists("week3_results.json"),
    "week2_processed_news.json exists":os.path.exists("week2_processed_news.json"),
    "data/raw_articles.json exists":   os.path.exists("data/raw_articles.json"),
    "requirements.txt exists":         os.path.exists("requirements.txt"),
}

print("Week 4 Deliverables Checklist")
print("=" * 45)
all_good = True
for item, status in checks.items():
    icon = "✔" if status else "✘"
    print(f"  {icon}  {item}")
    if not status:
        all_good = False

print("=" * 45)
if all_good:
    print("All checks passed.  Project complete! 🎉")
else:
    print("Some checks failed. See above.")
